In [1]:
import torch
import torch.nn.functional as F
import torch.nn.init as init
import matplotlib.pyplot as plt
import random

In [2]:
random.seed(42)
words = open("names.txt", "r").read().splitlines()
random.shuffle(words)
def stoi(c):
    if c == '.': return 26
    return ord(c)-ord('a')
def itos(i):
    if i == 26: return '.'
    return chr(ord('a')+i)

In [3]:
chunk_size = 3
emb_dim = 10
batch_size = 32
n_in = chunk_size*emb_dim
n_outs = [200, 27]

In [4]:
xs = []
ys = []
for word in words:
    context = [26]*chunk_size
    word = word + '.'
    for i in range(len(word)):
        
        xs.append(context)
        ys.append(stoi(word[i]))
        context = context[1:] + [stoi(word[i])]
xs = torch.tensor(xs)
ys = torch.tensor(ys)
xs.shape, ys.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [5]:
n1 = int(xs.shape[0]*0.8)
n2 = int(xs.shape[0]*0.9)
xdev = xs[n1:n2]
ydev = ys[n1:n2]
xtest = xs[n2:]
ytest = ys[n2:]
xs = xs[:n1]
ys = ys[:n1]

In [6]:
C = torch.randn((27,emb_dim))
W1 = torch.randn(n_in, n_outs[0]) * ((5/3) / n_in**0.5)
W2 = torch.randn(n_outs[0], n_outs[1]) * 0.01
B2 = torch.zeros(n_outs[1])
bngain = torch.ones(1, n_outs[0])
bnbias = torch.zeros(1, n_outs[0])
parameters = [C,W1,W2,B2,bngain,bnbias]
for param in parameters:
    param.requires_grad=True

In [ ]:
losses = []
for i in range(200000):
    batch = torch.randint(0, xs.shape[0], (batch_size, ))
    x = xs[batch]
    y = ys[batch]
    emb = C[x].view(-1, n_in) 
    hpre = emb @ W1
    # batch x outs[0]
    hpre = bngain*(hpre-hpre.mean(0, keepdim=True))/hpre.std(0, keepdim=True)+bnbias
    h = torch.tanh(hpre)
    logits = h @ W2 + B2
    loss = F.cross_entropy(logits, y)
    for param in parameters: param.grad = None
    loss.backward()
    for param in parameters: param.data -= (0.1 if i < 100000 else 0.01) * param.grad
    losses.append(loss.item())
    if i % 1000 == 0: print(f"step: {i}, loss {loss.item()}")
plt.plot(losses)


step: 0, loss 3.3007893562316895
step: 1000, loss 2.5699920654296875


In [ ]:
emb = C[xs].view(-1, n_in)
hpre = emb @ W1
bnmean = hpre.mean(0,keepdim=True)
bnstd = hpre.std(0,keepdim=True)
bnmean,bnstd

(tensor([[ 0.7896, -0.7520, -0.0618,  0.5284, -0.5843, -0.0418, -1.0367, -0.9856,
          -0.2547,  0.4792, -0.4853, -0.0072,  0.3763, -0.1938,  1.1739,  1.4632,
           0.7967, -0.4247,  0.5228, -1.4860,  0.2804,  0.2926,  0.0041, -0.3134,
          -0.5990,  0.2496, -0.3659, -0.5874, -0.0583, -0.6999, -0.6949,  0.2659,
           0.4358,  0.6310, -1.8206, -0.1863, -0.1164,  0.2120, -0.5466,  0.8755,
           1.3934,  1.1187, -0.4987,  1.1103,  0.8385, -0.1490, -0.3641, -0.8596,
          -0.1952, -0.0740,  0.4064, -1.0750,  0.7216, -0.7583,  0.0214,  0.8278,
          -0.4208,  0.4186, -0.2278, -1.1894,  0.0065, -0.7279, -0.2500,  0.8590,
           0.7681,  0.4303, -0.0810,  0.2763, -0.2559,  1.3544,  0.1923, -0.2842,
          -1.2079,  0.5416, -0.4129, -0.6485, -0.7474,  0.1464,  0.9811, -0.8111,
          -1.0067,  0.5982,  0.2884, -0.9894,  0.0572,  0.7019, -0.2504, -1.1380,
          -0.6922, -0.6082, -1.4585,  0.3665,  0.3381, -0.1212, -0.3213, -1.3111,
           0.174

In [ ]:
emb = C[xdev].view(-1, n_in) 
hpre = emb @ W1
hpre = bngain*(hpre-bnmean)/bnstd + bnbias
h = torch.tanh(hpre)
logits = h @ W2 + B2
loss = F.cross_entropy(logits, ydev)
print(loss.data)
emb = C[xtest].view(-1, n_in) 
hpre = emb @ W1
hpre = bngain*(hpre-bnmean)/bnstd + bnbias
h = torch.tanh(hpre)
logits = h @ W2 + B2
loss = F.cross_entropy(logits, ytest)
print(loss.data)


tensor(2.3205)
tensor(2.3514)


In [ ]:
for i in range(10):
    name = ""
    context = [26]*chunk_size
    while True:
        emb = C[(context, )].view(-1, emb_dim*chunk_size)
        hpre = emb @ W1
        hpre = bngain*(hpre-bnmean)/bnstd + bnbias
        h = torch.tanh(hpre)
        logits = h @ W2 + B2
        probs = F.softmax(logits, dim = 1)
        idx = torch.multinomial(probs, num_samples=1)
        name+=itos(idx)
        context = context[1:] + [idx]
        if idx == 26: break
    print(name)

soriadgy.
mirolan.
adelanah.
shuus.
darak.
rausti.
evion.
zakeelania.
cecer.
merbenna.
